In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import numpy as np
import pandas as pd
import torch

# from hyperopt import fmin, tpe, hp, Trials, rand
import xgboost as xgb
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/our_codes


In [ ]:
slice0data = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "slice0data.pkl")
)
slice1data = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "slice1data.pkl")
)
slice2data = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "slice2data.pkl")
)
slice3data = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "slice3data.pkl")
)
slice4data = pd.read_pickle(
    join(CURRENT_DIR, "..", "data", "our_data", "slice4data.pkl")
)

In [ ]:
slice0model = pd.read_pickle(
    os.path.join(CURRENT_DIR, "..", "data", "our_data", "slice0model.dat")
)
slice1model = pd.read_pickle(
    os.path.join(CURRENT_DIR, "..", "data", "our_data", "slice1model.dat")
)
slice2model = pd.read_pickle(
    os.path.join(CURRENT_DIR, "..", "data", "our_data", "slice2model.dat")
)
slice3model = pd.read_pickle(
    os.path.join(CURRENT_DIR, "..", "data", "our_data", "slice3model.dat")
)
slice4model = pd.read_pickle(
    os.path.join(CURRENT_DIR, "..", "data", "our_data", "slice4model.dat")
)

In [ ]:
def create_input_and_output_data(df):
    X = ()
    y = ()
    for ind in df.index:
        emb = df["ESM1b"][ind]
        ecfp = np.array(list(df["ECFP"][ind])).astype(int)

        X = X + (np.concatenate([ecfp, emb]),)
        y = y + (df["Binding"][ind],)

    return (X, y)


feature_names = ["ECFP_" + str(i) for i in range(1024)]
feature_names = feature_names + ["ESM1b_" + str(i) for i in range(1280)]
slice0data_X, slice0data_y = create_input_and_output_data(df=slice0data)
slice1data_X, slice1data_y = create_input_and_output_data(df=slice1data)
slice2data_X, slice2data_y = create_input_and_output_data(df=slice2data)
slice3data_X, slice3data_y = create_input_and_output_data(df=slice3data)
slice4data_X, slice4data_y = create_input_and_output_data(df=slice4data)

In [ ]:
slicedatas = [slice0data, slice1data, slice2data, slice3data, slice4data]
slicedatas_x = [slice0data_X, slice1data_X, slice2data_X, slice3data_X, slice4data_X]
slicedatas_y = [slice0data_y, slice1data_y, slice2data_y, slice3data_y, slice4data_y]
slicemodels = [slice0model, slice1model, slice2model, slice3model, slice4model]

In [ ]:
for i in range(len(slicedatas)):
    bst = slicemodels[i]
    dwant = xgb.DMatrix(
        np.array(slicedatas_x[i]),
        label=np.array(slicedatas_y[i]),
        feature_names=feature_names,
    )
    y_test_pred = bst.predict(dwant)
    slicedatas[i]["scores"] = y_test_pred

In [ ]:
merged_data = pd.concat(slicedatas, ignore_index=True)
sorted_data = merged_data.sort_values(
    by=["substrate", "scores"], ascending=[True, False]
)
sorted_data["ranking"] = sorted_data.groupby("substrate").cumcount() + 1
sorted_data.to_pickle(our_data + "5foldsdata.pkl")

In [ ]:
slice0data[slice0data["enzyme"] == "CYP82P3"]

,enzyme,substrate,Binding,ESM1b,ECFP,scores


In [ ]:
slice4data[slice4data["enzyme"] == "CYP82P3"]

,enzyme,substrate,Binding,ESM1b,ECFP,scores
19908,CYP82P3,GEI,0.0,"[-0.009233757853507996, 0.12540318071842194, -...",0000000000000000000000000000000001000000000010...,0.143611
19909,CYP82P3,CAT,0.0,"[-0.009233757853507996, 0.12540318071842194, -...",0100000010000000000000000000010001001000000000...,0.116287
19910,CYP82P3,BAM,0.0,"[-0.009233757853507996, 0.12540318071842194, -...",0000000000000000000000000000000101001000000000...,0.137926
19911,CYP82P3,NME,0.0,"[-0.009233757853507996, 0.12540318071842194, -...",0001000000000000000000000100000001000000100000...,0.233580
19912,CYP82P3,DOC,0.0,"[-0.009233757853507996, 0.12540318071842194, -...",0100000010000000000000000000000001001000000000...,0.084804
...,...,...,...,...,...,...
20140,CYP82P3,COL,0.0,"[-0.009233757853507996, 0.12540318071842194, -...",0100000000000000000000000100000001001000000000...,0.043127
20141,CYP82P3,OME,0.0,"[-0.009233757853507996, 0.12540318071842194, -...",0000000000000000000001000000000001000000100000...,0.106388
20142,CYP82P3,GGE,0.0,"[-0.009233757853507996, 0.12540318071842194, -...",0000000000000000000000000000000001000000000010...,0.219523
20143,CYP82P3,biochaninA,0.0,"[-0.009233757853507996, 0.12540318071842194, -...",0000000000000000000000000000000001000000000000...,0.212996
